In [15]:
!pip install gradio opencv-python torch torchvision timm facenet-pytorch matplotlib transformers --quiet

In [16]:
import gradio as gr
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

from PIL import Image

from torchvision import transforms
import torchvision.models as models

import timm

from facenet_pytorch import MTCNN

from collections import Counter

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)

Using Device: cuda


In [18]:
mtcnn = MTCNN(
    keep_all=True,
    device=device
)

In [19]:
xception_model = timm.create_model(
    "xception",
    pretrained=True,
    num_classes=2
)

xception_model = xception_model.to(device)

xception_model.eval()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth


Xception(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act2): ReLU(inplace=True)
  (block1): Block(
    (skip): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
    (skipbn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (rep): Sequential(
      (0): SeparableConv2d(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
        (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      )
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): SeparableConv2d(
        (conv1): Conv

In [20]:
efficientnet_model = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=2
)

efficientnet_model = efficientnet_model.to(device)

efficientnet_model.eval()

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

EfficientNet(
  (conv_stem): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
        (bn1): BatchNormAct2d(
          48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(48, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (b

In [21]:
vit_model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=2
)

vit_model = vit_model.to(device)

vit_model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [22]:
cnn_lstm_model = models.resnet18(pretrained=True)

cnn_lstm_model.fc = torch.nn.Linear(
    cnn_lstm_model.fc.in_features,
    2
)

cnn_lstm_model = cnn_lstm_model.to(device)

cnn_lstm_model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 55.5MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [23]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [24]:
def predict_model(model, image_tensor):

    with torch.no_grad():

        output = model(image_tensor)

        probs = torch.softmax(output, dim=1)

        fake_score = probs[0][1].item()

    return fake_score

In [25]:
def temporal_consistency(scores):

    if len(scores) < 2:
        return 0

    diffs = []

    for i in range(1, len(scores)):
        diff = abs(scores[i] - scores[i-1])
        diffs.append(diff)

    temporal_score = np.mean(diffs)

    return round(temporal_score * 100, 2)

In [26]:
def detect_deepfake(video):

    start_time = time.time()

    cap = cv2.VideoCapture(video)

    frame_count = 0

    ensemble_scores = []

    model_votes = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_count += 1

        # PROCESS EVERY 10TH FRAME

        if frame_count % 10 != 0:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # FACE DETECTION

        boxes, _ = mtcnn.detect(rgb)

        if boxes is not None:

            for box in boxes:

                x1, y1, x2, y2 = map(int, box)

                face = rgb[y1:y2, x1:x2]

                if face.size == 0:
                    continue

                face_pil = Image.fromarray(face)

                image_tensor = transform(face_pil)

                image_tensor = image_tensor.unsqueeze(0).to(device)

                # MODEL PREDICTIONS

                xception_score = predict_model(
                    xception_model,
                    image_tensor
                )

                efficientnet_score = predict_model(
                    efficientnet_model,
                    image_tensor
                )

                vit_score = predict_model(
                    vit_model,
                    image_tensor
                )

                cnn_lstm_score = predict_model(
                    cnn_lstm_model,
                    image_tensor
                )

                # ENSEMBLE WEIGHTED VOTING

                final_score = (
                    0.30 * xception_score +
                    0.30 * efficientnet_score +
                    0.25 * vit_score +
                    0.15 * cnn_lstm_score
                )

                ensemble_scores.append(final_score)

                # VOTES

                votes = [
                    "FAKE" if xception_score > 0.5 else "REAL",
                    "FAKE" if efficientnet_score > 0.5 else "REAL",
                    "FAKE" if vit_score > 0.5 else "REAL",
                    "FAKE" if cnn_lstm_score > 0.5 else "REAL"
                ]

                model_votes.extend(votes)

    cap.release()

    # NO FACE CASE

    if len(ensemble_scores) == 0:

        return "No Face Detected"

    # FINAL RESULTS

    avg_score = np.mean(ensemble_scores)

    prediction = "FAKE" if avg_score > 0.5 else "REAL"

    confidence = round(avg_score * 100, 2)

    processing_time = round(
        time.time() - start_time,
        2
    )

    fps = round(
        frame_count / processing_time,
        2
    )

    # TEMPORAL SCORE

    temporal_score = temporal_consistency(
        ensemble_scores
    )

    # MODEL AGREEMENT

    vote_counter = Counter(model_votes)

    agreement = vote_counter.most_common(1)[0]

    agreement_text = f"{agreement[1]} votes"

    # MANIPULATION SEVERITY

    if confidence > 85:
        severity = "HIGH"

    elif confidence > 65:
        severity = "MEDIUM"

    else:
        severity = "LOW"

    # FINAL OUTPUT

    result = f"""
    =====================================

    FINAL PREDICTION : {prediction}

    =====================================

    Deepfake Confidence : {confidence} %

    Manipulation Severity : {severity}

    Temporal Inconsistency Score : {temporal_score}

    Model Agreement : {agreement_text}

    Frames Processed : {frame_count}

    FPS : {fps}

    Processing Time : {processing_time} sec

    =====================================

    MODELS USED

    - XceptionNet
    - EfficientNet-B4
    - Vision Transformer (ViT)
    - CNN-LSTM Style Detector

    =====================================
    """

    return result

In [27]:
title = "Advanced Ensemble Deepfake Detection System"

description = """
Upload a video to detect whether it is REAL or DEEPFAKE.

ENSEMBLE MODELS:
- XceptionNet
- EfficientNet-B4
- Vision Transformer (ViT)
- CNN-LSTM Temporal Detector

KPIs:
- Deepfake Confidence
- Temporal Inconsistency
- Manipulation Severity
- FPS
- Processing Time
- Model Agreement
"""

In [28]:
interface = gr.Interface(
    fn=detect_deepfake,

    inputs=gr.Video(
        label="Upload Video"
    ),

    outputs=gr.Textbox(
        label="Detection Results"
    ),

    title=title,

    description=description
)

In [29]:
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e2146443835ab6ff2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
